# AstroCLIMB — full-vocabulary ensemble + restricted-loss blend

CPU-only notebook that blends the current three-seed full-vocabulary Qwen3-VL-8B ensemble with the completed Qwen3-VL-8B restricted-loss model:

$$p=\lambda p_{\mathrm{full}}+(1-\lambda)p_{\mathrm{restricted}}.$$

Attach the three-seed ensemble `submission_probabilities.csv` and either the restricted-loss `submission_probabilities.csv` or its `probabilities_rank*.csv` inference shards. Hard one-hot submissions are intentionally rejected because they discard confidence information.

For principled selection, provide both systems' probabilities on the same fixed validation IDs and the validation labels. Do not use `solution.csv` or leaderboard scores to select the weight.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

TARGETS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
PROB_COLS = [f'p_{name}' for name in TARGETS]
FULL_WEIGHTS = [0.25, 0.50, 0.75]
MINIMUM_GAIN = 0.005
BOOTSTRAP_REPLICATES = 1000

# Optional explicit test paths. The restricted path may be a merged CSV or shard directory.
FULL_TEST_PROBABILITIES = None
RESTRICTED_TEST_PROBABILITIES = None

# Optional fixed-validation inputs: provide all three or leave all as None.
FULL_VALIDATION_PROBABILITIES = None
RESTRICTED_VALIDATION_PROBABILITIES = None
VALIDATION_LABELS = None

# Without validation inputs, set this only to an independently selected weight.
# Leave None to write three candidate directories but no canonical submission.csv.
SELECTED_FULL_WEIGHT = None

WORK_ROOT = Path('/kaggle/working/astroclimb_full_restricted_blend') if Path('/kaggle/working').exists() else Path('./astroclimb_full_restricted_blend')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [p for p in [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')] if p.exists()]
print('Output:', WORK_ROOT)


## Locate and validate raw probability artifacts


In [ ]:
def probability_files():
    files = set()
    for root in SEARCH_ROOTS:
        files.update(root.rglob('submission_probabilities.csv'))
    return sorted(files, key=lambda p: (len(str(p)), str(p)))

def discover_full_source():
    matches = [p for p in probability_files() if 'three_seed' in str(p).lower() or 'three-seed' in str(p).lower()]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one three-seed probability file, found {len(matches)}. Set FULL_TEST_PROBABILITIES. Candidates: {matches}')
    return matches[0]

def discover_restricted_source():
    merged = [p for p in probability_files() if 'restricted' in str(p).lower()]
    if len(merged) == 1:
        return merged[0]
    shard_dirs = set()
    for root in SEARCH_ROOTS:
        for rank0 in root.rglob('probabilities_rank0.csv'):
            if 'restricted' in str(rank0).lower() and rank0.with_name('probabilities_rank1.csv').exists():
                shard_dirs.add(rank0.parent)
    if len(shard_dirs) == 1:
        return next(iter(shard_dirs))
    raise RuntimeError(
        'Raw restricted probabilities were not uniquely discoverable. Attach the restricted Kaggle output '
        'or set RESTRICTED_TEST_PROBABILITIES to a merged CSV or shard directory. '
        f'Merged candidates: {merged}; shard directories: {sorted(shard_dirs)}'
    )

def load_probabilities(source, name):
    source = Path(source)
    if source.name == 'submission.csv':
        raise ValueError(f'{name}: hard one-hot submission.csv cannot be used')
    if source.is_dir():
        shards = sorted(source.glob('probabilities_rank*.csv'))
        if not shards:
            raise FileNotFoundError(f'{name}: no probability shards in {source}')
        frame = pd.concat([pd.read_csv(p, dtype={'id': str}) for p in shards], ignore_index=True)
        print(f'{name}: merged {len(shards)} shards from {source}')
    else:
        frame = pd.read_csv(source, dtype={'id': str})
        print(f'{name}: loaded {source}')
    required = ['id', *PROB_COLS]
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f'{name}: missing raw probability columns {sorted(missing)}')
    frame = frame[required].copy()
    if frame['id'].duplicated().any():
        raise ValueError(f'{name}: duplicate IDs')
    values = frame[PROB_COLS].to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f'{name}: invalid probability values')
    totals = values.sum(axis=1, keepdims=True)
    if (totals <= 0).any():
        raise ValueError(f'{name}: non-positive probability mass')
    frame.loc[:, PROB_COLS] = values / totals
    return frame

full_source = Path(FULL_TEST_PROBABILITIES) if FULL_TEST_PROBABILITIES else discover_full_source()
restricted_source = Path(RESTRICTED_TEST_PROBABILITIES) if RESTRICTED_TEST_PROBABILITIES else discover_restricted_source()
full_test = load_probabilities(full_source, 'full ensemble test')
restricted_test = load_probabilities(restricted_source, 'restricted test')


## Align components and optionally select a blend weight on validation


In [ ]:
def align(full, restricted, name):
    full_ids, restricted_ids = set(full['id']), set(restricted['id'])
    if full_ids != restricted_ids:
        raise ValueError(f'{name}: ID mismatch; full-only={len(full_ids-restricted_ids)}, restricted-only={len(restricted_ids-full_ids)}')
    restricted = restricted.set_index('id').loc[full['id']].reset_index()
    assert restricted['id'].tolist() == full['id'].tolist()
    return full.reset_index(drop=True), restricted.reset_index(drop=True)

def blend(full, restricted, weight):
    a = full[PROB_COLS].to_numpy(dtype=np.float64)
    b = restricted[PROB_COLS].to_numpy(dtype=np.float64)
    values = weight * a + (1.0 - weight) * b
    return values / values.sum(axis=1, keepdims=True)

def macro_f1(y_true, y_pred):
    scores = []
    for c in range(4):
        truth, predicted = y_true == c, y_pred == c
        tp = np.logical_and(truth, predicted).sum()
        fp = np.logical_and(~truth, predicted).sum()
        fn = np.logical_and(truth, ~predicted).sum()
        denominator = 2 * tp + fp + fn
        scores.append(0.0 if denominator == 0 else 2 * tp / denominator)
    return float(np.mean(scores))

def load_labels(path):
    path = Path(path)
    if path.name.lower() == 'solution.csv':
        raise ValueError('Kaggle solution.csv must not be used for weight selection')
    frame = pd.read_csv(path, dtype={'id': str})
    missing = {'id', *TARGETS} - set(frame.columns)
    if missing:
        raise ValueError(f'Validation labels missing columns: {sorted(missing)}')
    matrix = frame[TARGETS].to_numpy(dtype=np.int64)
    if not np.isin(matrix, [0, 1]).all() or not (matrix.sum(axis=1) == 1).all():
        raise ValueError('Validation labels are not one-hot')
    return pd.DataFrame({'id': frame['id'], 'label': matrix.argmax(axis=1)})

full_test, restricted_test = align(full_test, restricted_test, 'test')
assert len(full_test) == 10000, f'Expected 10,000 test rows, got {len(full_test)}'

validation_paths = [FULL_VALIDATION_PROBABILITIES, RESTRICTED_VALIDATION_PROBABILITIES, VALIDATION_LABELS]
if any(p is not None for p in validation_paths) and not all(p is not None for p in validation_paths):
    raise ValueError('Provide all three validation paths or leave all three as None')

selected_weight = SELECTED_FULL_WEIGHT
passed_gate = None
if all(p is not None for p in validation_paths):
    full_val = load_probabilities(FULL_VALIDATION_PROBABILITIES, 'full validation')
    restricted_val = load_probabilities(RESTRICTED_VALIDATION_PROBABILITIES, 'restricted validation')
    full_val, restricted_val = align(full_val, restricted_val, 'validation')
    labels = load_labels(VALIDATION_LABELS).set_index('id').loc[full_val['id']]['label'].to_numpy()
    baseline_pred = full_val[PROB_COLS].to_numpy().argmax(axis=1)
    baseline_score = macro_f1(labels, baseline_pred)
    rows = []
    for weight in [0.0, *FULL_WEIGHTS, 1.0]:
        score = macro_f1(labels, blend(full_val, restricted_val, weight).argmax(axis=1))
        rows.append({'full_weight': weight, 'restricted_weight': 1-weight, 'macro_f1': score, 'gain_vs_full': score-baseline_score})
    results = pd.DataFrame(rows)
    display(results)
    best = results[results['full_weight'].isin(FULL_WEIGHTS)].sort_values(['macro_f1', 'full_weight'], ascending=[False, False]).iloc[0]
    selected_weight = float(best['full_weight'])
    selected_pred = blend(full_val, restricted_val, selected_weight).argmax(axis=1)
    rng, deltas = np.random.default_rng(42), []
    for _ in range(BOOTSTRAP_REPLICATES):
        idx = rng.integers(0, len(labels), size=len(labels))
        deltas.append(macro_f1(labels[idx], selected_pred[idx]) - macro_f1(labels[idx], baseline_pred[idx]))
    lower, upper = np.quantile(deltas, [0.025, 0.975])
    gain = float(best['gain_vs_full'])
    passed_gate = gain >= MINIMUM_GAIN and lower > 0
    print(f'Selected full weight={selected_weight:.2f}; gain={gain:.6f}; bootstrap 95% CI=[{lower:.6f}, {upper:.6f}]')
    print('Promotion gate passed:', passed_gate)
elif selected_weight is not None:
    if selected_weight not in FULL_WEIGHTS:
        raise ValueError(f'SELECTED_FULL_WEIGHT must be one of {FULL_WEIGHTS}')
    print(f'Using independently selected full-model weight: {selected_weight:.2f}')
else:
    print('No selection inputs supplied. The notebook will write candidates but no canonical submission.csv.')


## Write candidate blends and the gated final submission


In [ ]:
def write_outputs(weight, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    values = blend(full_test, restricted_test, weight)
    probabilities = pd.DataFrame(values, columns=PROB_COLS)
    probabilities.insert(0, 'id', full_test['id'])
    predictions = values.argmax(axis=1)
    submission = pd.DataFrame({'id': full_test['id']})
    for class_index, name in enumerate(TARGETS):
        submission[name] = (predictions == class_index).astype(np.int8)
    assert len(submission) == 10000 and submission['id'].is_unique
    assert submission.columns.tolist() == ['id', *TARGETS]
    assert submission[TARGETS].isin([0, 1]).all().all()
    assert (submission[TARGETS].sum(axis=1) == 1).all()
    assert np.allclose(values.sum(axis=1), 1.0, atol=1e-12)
    probabilities.to_csv(output_dir / 'submission_probabilities.csv', index=False, lineterminator='\n')
    submission.to_csv(output_dir / 'submission.csv', index=False, lineterminator='\n')
    return submission

manifest = []
for weight in FULL_WEIGHTS:
    output_dir = WORK_ROOT / f'full_{weight:.2f}_restricted_{1-weight:.2f}'
    candidate = write_outputs(weight, output_dir)
    counts = candidate[TARGETS].sum().to_dict()
    manifest.append({'full_weight': weight, 'restricted_weight': 1-weight, 'directory': str(output_dir), **counts})
display(pd.DataFrame(manifest))

if selected_weight is not None and passed_gate is not False:
    selected = write_outputs(selected_weight, WORK_ROOT)
    metadata = {
        'full_weight': selected_weight,
        'restricted_weight': 1-selected_weight,
        'selected_with_validation': all(p is not None for p in validation_paths),
        'promotion_gate_passed': passed_gate,
        'full_test_source': str(full_source),
        'restricted_test_source': str(restricted_source),
    }
    (WORK_ROOT / 'blend_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
    print('Selected submission:', WORK_ROOT / 'submission.csv')
    print('Selected probabilities:', WORK_ROOT / 'submission_probabilities.csv')
elif passed_gate is False:
    print('The best tested blend failed the validation gate; no canonical submission.csv was written.')
else:
    print('Candidate outputs were written in weight-specific directories. Supply validation inputs or SELECTED_FULL_WEIGHT to create canonical outputs.')
